In [31]:
# libraries
import pandas as pd
import requests
import time

In [32]:
# only pull necessary columns
cols_to_use = [
    "Chromosome",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "ClinicalSignificance",
    "ReviewStatus",
    "NumberSubmitters"
]

clinvar_df = pd.read_csv(
    "variant_summary.txt",
    sep="\t",
    usecols=cols_to_use,
    dtype=str,
    low_memory=True,
    engine="c"
)

clinvar_df.columns = clinvar_df.columns.str.strip().str.replace("#", "")

In [33]:
# convert to variant id format
clinvar_df["VariantID"] = (
    clinvar_df["Chromosome"].astype(str) + "-" +
    clinvar_df["PositionVCF"].astype(str) + "-" +
    clinvar_df["ReferenceAlleleVCF"].astype(str) + "-" +
    clinvar_df["AlternateAlleleVCF"].astype(str)
)

In [34]:
# can add more col as needed
clinvar_subset = clinvar_df[["VariantID", "ClinicalSignificance", "ReviewStatus", "NumberSubmitters"]].dropna()

In [35]:
# read input variant file
input_df = pd.read_csv("test_variants.txt", dtype=str)

print(input_df.columns.tolist())

['Variant ID']


In [36]:
input_df = input_df.rename(columns={"Variant ID": "VariantID"})

In [37]:
input_df["VariantID"] = input_df["VariantID"].str.replace('"', '').str.strip()

In [38]:
# merge both
annotation_df = input_df.merge(
    clinvar_subset,
    on="VariantID",
    how="left"
)

In [16]:
annotation_df.head()

,VariantID,ClinicalSignificance,ReviewStatus,NumberSubmitters
0,1-94471075-A-G,Benign,"criteria provided, multiple submitters, no con...",13
1,1-94473807-C-T,Pathogenic,reviewed by expert panel,61
2,1-94473845-T-C,Benign/Likely benign,"criteria provided, multiple submitters, no con...",14
3,1-225492710-C-T,NaN,NaN,NaN
4,1-225586901-C-A,NaN,NaN,NaN


In [12]:
annotation_df.to_csv("results.csv", index=False)

### gnomAD

### Modify it to pull exome, genome and allele number

In [41]:
# select refernce genome
genome = "hg19"  # change to hg38 if needed

if genome.lower() in ["hg19", "grch37"]:
    dataset = "gnomad_r2_1"
else:
    dataset = "gnomad_r3"

In [42]:
import requests
import time

In [43]:
def fetch_block(variant_id, dataset, block, retries=3):
    """
    Helper function to fetch genome or exome block separately
    """
    url = "https://gnomad.broadinstitute.org/api"

    query = {
        "query": f"""
        {{
          variant(variantId: "{variant_id}", dataset: {dataset}) {{
            {block} {{
              af
              ac
              an
            }}
          }}
        }}
        """
    }

    for _ in range(retries):
        try:
            r = requests.post(url, json=query)
            data = r.json()

            if "errors" in data:
                return None

            variant = data.get("data", {}).get("variant")
            if not variant:
                return None

            return variant.get(block)

        except:
            time.sleep(0.5)

    return None


def fetch_af(variant_id, dataset):
    """
    Fetch genome AF, exome AF, and total AF
    """

    genome = fetch_block(variant_id, dataset, "genome")
    exome = fetch_block(variant_id, dataset, "exome")

    # Extract values safely
    genome_af = genome.get("af") if genome else None
    exome_af = exome.get("af") if exome else None

    genome_ac = genome.get("ac") if genome else 0
    genome_an = genome.get("an") if genome else 0

    exome_ac = exome.get("ac") if exome else 0
    exome_an = exome.get("an") if exome else 0

    # Compute total AF using AC/AN (correct way)
    total_ac = (genome_ac or 0) + (exome_ac or 0)
    total_an = (genome_an or 0) + (exome_an or 0)

    total_af = total_ac / total_an if total_an else None

    return {
        "genome_af": genome_af,
        "exome_af": exome_af,
        "total_af": total_af
    }


# ===== APPLY TO DATAFRAME =====

def annotate_af(annotation_df, dataset):
    results = []

    for vid in annotation_df["VariantID"]:
        af = fetch_af(vid, dataset)
        results.append(af)

        time.sleep(1.0)  # safer rate limit

    # Expand into columns
    af_df = pd.DataFrame(results)
    annotation_df = pd.concat([annotation_df, af_df], axis=1)

    return annotation_df

In [44]:
# case that kept failing, but should work now
fetch_af("1-228491538-C-T", dataset)

{'genome_af': 0.004559948979591837,
 'exome_af': 0.0011755077150216245,
 'total_af': 0.0015537357366346655}

In [46]:
af_list = []

for vid in annotation_df["VariantID"]:
    af = fetch_af(vid, dataset)
    af_list.append(af)

    time.sleep(7.0)  # 6sec/query is API limit, dont get blocked.

# Convert list of dicts → DataFrame
af_df = pd.DataFrame(af_list)

# Merge with original dataframe
annotation_df = pd.concat([annotation_df.reset_index(drop=True), af_df], axis=1)

In [47]:
annotation_df

,VariantID,ClinicalSignificance,ReviewStatus,NumberSubmitters,genome_af,exome_af,total_af
0,1-94471075-A-G,Benign,"criteria provided, multiple submitters, no con...",13,NaN,NaN,NaN
1,1-94473807-C-T,Pathogenic,reviewed by expert panel,61,NaN,NaN,NaN
2,1-94473845-T-C,Benign/Likely benign,"criteria provided, multiple submitters, no con...",14,NaN,0.172861,0.172861
3,1-225492710-C-T,NaN,NaN,NaN,0.010955,0.002196,0.003702
4,1-225586901-C-A,NaN,NaN,NaN,0.004045,0.000906,0.001431
5,1-228461611-C-T,Benign/Likely benign,"criteria provided, multiple submitters, no con...",4,0.004937,0.001170,0.001619
6,1-228527749-C-T,Benign,"criteria provided, multiple submitters, no con...",3,0.007944,0.002720,0.003304
7,1-228476484-G-A,Benign/Likely benign,"criteria provided, multiple submitters, no con...",4,0.007518,NaN,0.007518
8,1-228480317-C-T,Benign/Likely benign,"criteria provided, multiple submitters, no con...",5,NaN,NaN,NaN
9,1-228491538-C-T,Benign,no assertion criteria provided,1,NaN,0.001176,0.001176
